In [253]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [254]:
def sigmoid(x):
        return 1 / (1 + np.exp(-x))
def sigmoid_derivative(x):
        return x * (1 - x)

In [255]:
def network_initialize(input_feauter, n_hiddens=[3], output_feature=1):
    network = []
    network.append({'weights':np.random.rand(input_feauter+1, n_hiddens[0])})
    for i in range(1, len(n_hiddens)):
        network.append({'weights':np.random.rand(n_hiddens[i-1]+1, n_hiddens[i])})
    network.append({'weights':np.random.rand(n_hiddens[-1]+1, output_feature)})
    return network

# network_initialize(3, [2,2], 1)

In [256]:
def feedforward(network, X):
    net = np.dot(X, network[0]['weights'][:-1]) + network[0]['weights'][-1]
    network[0]['net'] = net
    network[0]['out'] = sigmoid(net)
    for i in range(1, len(network)):
        net = np.dot(network[i - 1]['out'], network[i]['weights'][:-1]) + network[i]['weights'][-1]
        network[i]['net'] = net
        network[i]['out'] = np.array(sigmoid(net)).reshape(len(X))
    return network


# network = network_initialize(1, [2], 1)
# print(network)
# network = feedforward(network, [[2],[3]])
# print(network)

In [257]:
def backpropagation(network, X, y, learning_rate):
    # Forward pass
    network = feedforward(network, X)
    
    # Calculate error for the output layer
    error = y - network[-1]['out']
    delta_output = error * sigmoid_derivative(network[-1]['out'])
    # Update weights for the output layer

    network[-1]['weights'][:-1] += learning_rate * np.dot(network[-2]['out'].T, delta_output).reshape(-1,1)
    network[-1]['weights'][-1] += learning_rate * np.sum(delta_output, axis=0)  # Update bias
    
    # Calculate error for hidden layers and update weights
    for i in range(len(network) - 2, 0, -1):
        error = np.dot(delta_output, network[i + 1]['weights'][:-1].T)
        delta_hidden = error * sigmoid_derivative(network[i]['out'])
        network[i]['weights'][:-1] += learning_rate * np.dot(network[i - 1]['out'].T, delta_hidden).reshape(-1, 1)
        network[i]['weights'][-1] += learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
        delta_output = delta_hidden
    
    # Update weights for the first hidden layer
    delta_output = delta_output.reshape(-1, 1)
    error = np.dot(delta_output, network[1]['weights'][:-1].T)
    delta_hidden = error * sigmoid_derivative(network[0]['out'])
    network[0]['weights'][:-1] += learning_rate * np.dot(X.T, delta_hidden)
    network[0]['weights'][-1] += learning_rate * np.sum(delta_hidden, axis=0)  # Update bias
    
    return network

In [258]:
def train(X, y, epochs, learning_rate):
    network = network_initialize(input_feauter=X.shape[1], n_hiddens=[3, 3], output_feature=1)
    for epoch in range(epochs):
        backpropagation(network, X, y, learning_rate)
        if epoch % 100 == 0:
            print(f"epoch {epoch} finished")
    return network

def prediction(X, trained_network):
    y_pred = np.round(feedforward(trained_network, X)[-1]['out'])
    return y_pred

        
# X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
# y = np.array([[0], [1], [1], [0]])
# trained_network = train(X, y, 6000, .1)
# print(accuracy_score(y, prediction(X, trained_network)))

In [259]:
def load_data(): 
    train_loaded = np.load(f'train_data.npz')
    test_loaded = np.load(f'test_data.npz')
    
    train_x, train_y = train_loaded['x'].T, train_loaded['y'] 
    test_x, test_y = test_loaded['x'].T, test_loaded['y'] 

    # Standarize data
    train_x = train_x / 255.
    test_x = test_x / 255.

    # Shuffle data (no effect for full batch method)
    train_indices = np.random.permutation(train_x.shape[1])
    train_x = train_x[:, train_indices]
    train_y = train_y[train_indices]

    test_indices = np.random.permutation(test_x.shape[1])
    test_x = test_x[:, test_indices]
    test_y = test_y[test_indices]
    
    return train_x.T, train_y, test_x.T, test_y

train_x, train_y, test_x, test_y = load_data()
trained_network = train(train_x, train_y, 500, .1)
print(accuracy_score(test_y, prediction(test_x, trained_network)))

ValueError: cannot reshape array of size 6000 into shape (2000,)